# QLoRA Fine-Tuning: Tamil Grammar Correction Model

Fine-tune a Tamil-capable base model on the generated grammar correction dataset
using **QLoRA** (4-bit quantization + LoRA).

### Requirements
- Google Colab with **T4 GPU** (or local RTX 3050 6GB+)
- `train.jsonl` and `eval.jsonl` from the dataset generation step

### Pipeline
```
train.jsonl + eval.jsonl
  -> Format as Alpaca-style instruction pairs
  -> Load base model in 4-bit (QLoRA)
  -> Apply LoRA adapters
  -> Train with SFTTrainer
  -> Evaluate on eval set
  -> Export merged model
```

### Instructions
1. **Runtime -> Change runtime type -> T4 GPU**
2. Upload `train.jsonl` and `eval.jsonl` to Colab, or mount Google Drive.
3. Run cells sequentially.

---

## CELL 1 - Enable GPU

In [ ]:
!nvidia-smi

---

## CELL 2 - Install dependencies

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft trl datasets sentencepiece tqdm

---

## CELL 3 - Check versions

In [ ]:
import torch
import transformers
import accelerate
import peft
import trl

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Accelerate:', accelerate.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    mem = round(torch.cuda.get_device_properties(0).total_mem / 1024**3, 2)
    print(f'GPU memory: {mem} GB')
else:
    print('WARNING: No GPU detected!')

---

## CELL 4 - Configuration

In [ ]:
from pathlib import Path
import os
import json
import random

# ============================================================
# MODEL - Choose a Tamil-capable base model
# ============================================================
# Option A: Qwen3-8B (multilingual, good Tamil support)
MODEL_NAME = 'Qwen/Qwen3-8B'
#
# Option B: Tamil LLaMA 7B (existing project model)
# MODEL_NAME = 'abhinand/tamil-llama-7b-base-v0.1'
#
# Option C: TinyLlama 1.1B (smallest, fastest for testing)
# MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# ============================================================
# DATASET
# ============================================================
# Path to train.jsonl - adjust based on your Colab setup
TRAIN_FILE = Path('train.jsonl')
EVAL_FILE = Path('eval.jsonl')

# If files are on Google Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')
# TRAIN_FILE = Path('/content/drive/MyDrive/tamil_grammar_dataset/train.jsonl')
# EVAL_FILE = Path('/content/drive/MyDrive/tamil_grammar_dataset/eval.jsonl')

# ============================================================
# OUTPUT
# ============================================================
OUTPUT_DIR = Path('./qlora_tamil_grammar')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_DIR = OUTPUT_DIR / 'merged_model'
LORA_MODEL_DIR = OUTPUT_DIR / 'lora_adapter'

SEED = 42
random.seed(SEED)

print(f'Base model: {MODEL_NAME}')
print(f'Train file: {TRAIN_FILE}')
print(f'Eval file: {EVAL_FILE}')
print(f'Output dir: {OUTPUT_DIR}')

---

## CELL 5 - Upload dataset

Upload `train.jsonl` and `eval.jsonl` if they are not already in the Colab working directory.

In [ ]:
# Option 1: Upload from local machine
if not TRAIN_FILE.exists():
    print('train.jsonl not found. Upload it now:')
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        print(f'Uploaded: {fn}')
else:
    print(f'train.jsonl found: {TRAIN_FILE}')

if not EVAL_FILE.exists():
    print('eval.jsonl not found. Upload it now:')
    from google.colab import files
    uploaded = files.upload()
    for fn in uploaded.keys():
        print(f'Uploaded: {fn}')
else:
    print(f'eval.jsonl found: {EVAL_FILE}')

---

## CELL 6 - Load and format dataset

Converts the JSONL data into the chat/instruction format expected by the model.

In [ ]:
from datasets import load_dataset

# Load JSONL files
raw_train = load_dataset('json', data_files=str(TRAIN_FILE), split='train')
raw_eval = load_dataset('json', data_files=str(EVAL_FILE), split='train')

print(f'Train examples: {len(raw_train)}')
print(f'Eval examples: {len(raw_eval)}')
print()
print('Sample entry:')
print(json.dumps(raw_train[0], ensure_ascii=False, indent=2))

---

## CELL 7 - Format data for SFT training

Creates chat-format messages for each example.

In [ ]:
def format_example(example):
    """Format a single example into chat messages for SFT."""
    instruction = example.get('instruction', '')
    input_text = example.get('input', '')
    output_text = example.get('output', '')

    # Build the user message
    if input_text and input_text.strip():
        user_msg = f'{instruction}\n\n{input_text}'
    else:
        user_msg = instruction

    messages = [
        {'role': 'user', 'content': user_msg},
        {'role': 'assistant', 'content': output_text},
    ]

    return {'messages': messages}


# Apply formatting
train_dataset = raw_train.map(format_example, remove_columns=raw_train.column_names)
eval_dataset = raw_eval.map(format_example, remove_columns=raw_eval.column_names)

print('Formatted datasets ready.')
print()
print('Sample formatted entry:')
print(json.dumps(train_dataset[0]['messages'], ensure_ascii=False, indent=2))

---

## CELL 8 - Load tokenizer

In [ ]:
from transformers import AutoTokenizer

print(f'Loading tokenizer from {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side='right',
)

# Set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f'Set pad_token to eos_token: {tokenizer.eos_token}')

print(f'Tokenizer vocab size: {len(tokenizer)}')
print(f'pad_token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})')
print(f'eos_token: {tokenizer.eos_token} (id={tokenizer.eos_token_id})')

---

## CELL 9 - Load model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_NAME} in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

model.config.use_cache = False

print('Model loaded!')
print(f'Model parameters: {model.num_parameters():,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

---

## CELL 10 - Check GPU memory after model load

In [ ]:
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f'GPU total:   {total:.2f} GB')
    print(f'GPU alloc:   {allocated:.2f} GB')
    print(f'GPU reserved: {reserved:.2f} GB')
    print(f'GPU free:    {total - allocated:.2f} GB')
    if total - allocated < 2.0:
        print('WARNING: Less than 2GB free. Reduce batch size or sequence length.')

---

## CELL 11 - Configure LoRA

Conservative LoRA settings suitable for T4 / RTX 3050.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # LoRA rank (8-64, lower = less VRAM)
    lora_alpha=32,                 # Alpha scaling (usually 2x rank)
    lora_dropout=0.05,             # Dropout for regularization
    target_modules=[
        'q_proj',                  # Query projection
        'k_proj',                  # Key projection
        'v_proj',                  # Value projection
        'o_proj',                  # Output projection
        'gate_proj',               # MLP gate
        'down_proj',               # MLP down
        'up_proj',                 # MLP up
    ],
    bias='none',
)

# Apply LoRA
model = get_peft_model(model, lora_config)

print('LoRA applied!')
model.print_trainable_parameters()

---

## CELL 12 - Training configuration

In [ ]:
from transformers import TrainingArguments

# ============================================================
# Adjust these based on your GPU:
#   T4 (16GB):    batch_size=4,  grad_accum=8,  seq_len=512
#   RTX 3050 (6GB): batch_size=2,  grad_accum=16, seq_len=512
# ============================================================

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),

    # Training schedule
    num_train_epochs=3,                      # 2-3 epochs for SFT
    per_device_train_batch_size=4,           # Reduce to 2 if OOM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,           # Effective batch = 4*8 = 32

    # Optimizer
    optim='paged_adamw_8bit',                # Memory-efficient optimizer
    learning_rate=2e-4,                      # Standard QLoRA LR
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,

    # Precision
    fp16=True,                               # Use bf16=True on A100
    bf16=False,

    # Memory optimization
    gradient_checkpointing=True,             # Save VRAM at cost of speed
    group_by_length=True,                    # Speed up by grouping similar lengths

    # Logging & saving
    logging_steps=25,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,                      # Keep only last 3 checkpoints
    eval_strategy='steps',
    eval_steps=200,

    # Hub (optional - set to True to push to HuggingFace)
    push_to_hub=False,

    # Misc
    seed=42,
    report_to='tensorboard',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
)

print('Training arguments configured.')
print(f'  Epochs: {training_args.num_train_epochs}')
print(f'  Batch size: {training_args.per_device_train_batch_size}')
print(f'  Grad accum: {training_args.gradient_accumulation_steps}')
eff_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f'  Effective batch: {eff_batch}')
print(f'  Learning rate: {training_args.learning_rate}')
print(f'  Max seq length: 512')

---

## CELL 13 - Initialize SFTTrainer

In [ ]:
from trl import SFTTrainer

MAX_SEQ_LENGTH = 512

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='messages',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,                           # Set True to pack short examples
)

print('SFTTrainer initialized.')
print(f'Trainable params: {trainer.model.print_trainable_parameters()}')

---

## CELL 14 - Train!

**Tip:** On T4, expect ~1-2 hours for 20K examples. On RTX 3050, slightly faster.

In [ ]:
print('Starting training...')
print('=' * 60)

train_result = trainer.train()

print('=' * 60)
print('Training complete!')
print()

# Print metrics
metrics = train_result.metrics
print(f'Train loss: {metrics.get("train_loss", "N/A")}')
print(f'Train runtime: {metrics.get("train_runtime", 0):.0f}s')
print(f'Train samples/s: {metrics.get("train_samples_per_second", 0):.2f}')

---

## CELL 15 - Save LoRA adapter

In [ ]:
# Save just the LoRA adapter (small, ~100MB)
trainer.model.save_pretrained(str(LORA_MODEL_DIR))
tokenizer.save_pretrained(str(LORA_MODEL_DIR))

print(f'LoRA adapter saved to: {LORA_MODEL_DIR}')

# Show file sizes
total_size = 0
for f in LORA_MODEL_DIR.iterdir():
    size_mb = f.stat().st_size / 1024**2
    total_size += size_mb
    print(f'  {f.name}: {size_mb:.1f} MB')
print(f'  Total: {total_size:.1f} MB')

---

## CELL 16 - Evaluate on eval set

In [ ]:
print('Running evaluation...')
eval_results = trainer.evaluate()

print()
print('Eval results:')
for k, v in eval_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

---

## CELL 17 - Test inference with fine-tuned model

In [ ]:
def test_inference(bad_sentence, max_new_tokens=200):
    """Test the fine-tuned model on a Tamil grammar correction task."""
    prompt = (
        'இந்த தமிழ் வாக்கியத்தில் உள்ள இலக்கணப் பிழையை திருத்தவும்.\n\n'
        f'{bad_sentence}'
    )

    messages = [{'role': 'user', 'content': prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True,
    )
    return response


# Test with sample sentences
test_sentences = [
    'அவன் நேற்று பள்ளிக்கு போகிறான்.',
    'நான் நேற்று சென்னைக்கு செல்கிறேன்.',
    'அவள் புத்தகத்தை படிக்கிறார்.',
]

for sent in test_sentences:
    print('=' * 60)
    print(f'INPUT:    {sent}')
    result = test_inference(sent)
    print(f'OUTPUT:   {result}')
    print()

---

## CELL 18 - Merge and export full model

This creates a standalone model (no LoRA) for easier deployment.

In [ ]:
import gc
from peft import PeftModel

print('Merging LoRA adapter into base model...')
print('This requires enough RAM for the full model.\n')

# Clear GPU memory
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

# Reload base model in float16 (not quantized)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)

# Load and merge LoRA
model = PeftModel.from_pretrained(base_model, str(LORA_MODEL_DIR))
model = model.merge_and_unload()

print('Model merged!')

# Save merged model
model.save_pretrained(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

print(f'Merged model saved to: {FINAL_MODEL_DIR}')

---

## CELL 19 - Zip and download models

In [ ]:
import shutil

# Zip LoRA adapter (small)
lora_zip = Path('/content/lora_adapter.zip')
if lora_zip.exists():
    lora_zip.unlink()
shutil.make_archive('/content/lora_adapter', 'zip', LORA_MODEL_DIR)
print(f'LoRA adapter zip: {lora_zip}')

# Zip merged model (large)
merged_zip = Path('/content/merged_model.zip')
if merged_zip.exists():
    merged_zip.unlink()
shutil.make_archive('/content/merged_model', 'zip', FINAL_MODEL_DIR)
print(f'Merged model zip: {merged_zip}')

# Download
from google.colab import files
print('\nDownloading LoRA adapter (recommended - small file)...')
files.download(str(lora_zip))

---

## CELL 20 - Save to Google Drive (recommended)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/tamil_grammar_model')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Copy LoRA adapter
shutil.copytree(str(LORA_MODEL_DIR), str(DRIVE_DIR / 'lora_adapter'), dirs_exist_ok=True)

# Copy merged model
shutil.copytree(str(FINAL_MODEL_DIR), str(DRIVE_DIR / 'merged_model'), dirs_exist_ok=True)

print(f'Saved to Google Drive: {DRIVE_DIR}')
print(f'  - lora_adapter/')
print(f'  - merged_model/')

---

## CELL 21 - Optional: Push to HuggingFace Hub

In [ ]:
# Uncomment to push to HuggingFace Hub
# from huggingface_hub import login
#
# login()  # Will prompt for token
#
# HF_REPO = 'your-username/tamil-grammar-qlora'
#
# # Push LoRA adapter
# model.push_to_hub(HF_REPO, use_temp_dir=False)
# tokenizer.push_to_hub(HF_REPO, use_temp_dir=False)
#
# print(f'Pushed to: https://huggingface.co/{HF_REPO}')

print('Skipped. Uncomment above to push to HuggingFace Hub.')

---

## Summary

### What was trained
- **Base model**: Qwen3-8B (or your chosen model)
- **Method**: QLoRA (4-bit NF4 + LoRA r=16)
- **Dataset**: Tamil grammar correction (~19K train, ~1K eval)
- **Output**: LoRA adapter + merged model

### Files produced
```
qlora_tamil_grammar/
  lora_adapter/      # LoRA weights (~100-200MB)
  merged_model/      # Full merged model (~16GB)
  checkpoints/       # Training checkpoints
```

### Next steps
1. **Evaluate** on the Tamil test suite from the project prompt.
2. **Convert to GGUF** for Ollama/LM Studio deployment:
   ```bash
   python convert_hf_to_gguf.py merged_model/ --outfile tamil-grammar.gguf
   ```
3. **Iterate**: adjust hyperparameters, add more data, try different base models.

### Hyperparameter tuning tips
- **OOM errors**: reduce `per_device_train_batch_size` to 2, increase `gradient_accumulation_steps`
- **Slow training**: increase `per_device_train_batch_size`, set `packing=True`
- **Overfitting**: increase `lora_dropout`, reduce `num_train_epochs`, add more data
- **Underfitting**: increase `num_train_epochs`, increase `learning_rate` slightly